# Home Automation System



In [1]:
"""
Home Automation System - Full Implementation with OOP Principles
Author: Devendra Chaudhari
Date: May 18, 2026
Course: Engineering Programming

This module implements a comprehensive smart home automation system using
inheritance hierarchies and composition patterns for maximum extensibility.
"""

import random
from datetime import datetime
import unittest

# ============================================================================
# STAGE 1: BASIC FRAMEWORK - Core Device Classes
# ============================================================================

class Device:
    """Abstract base class for all home automation devices."""
    def __init__(self, name):
        self.name = name
        self.status = self.get_status()
    def get_status(self):
        """Return the current device status or raise if undefined."""
        raise NotImplementedError()
    def is_accessible(self):
        """Return True when the device is currently available for use."""
        raise NotImplementedError()
    def reset(self):
        """Restore the device to its initial safe state."""
        raise NotImplementedError()
    def energy_usage(self):
        """Estimate the device's current energy consumption in watts."""
        return 0.0
    def __str__(self):
        return f"{self.name}: {self.status!s}"

class Sensor(Device):
    """Base class for all sensors. Implements failure-safe semantics (None = failure)."""
    def is_accessible(self):
        """Sensors are accessible when the status is not None."""
        return self.status is not None
    def reset(self):
        """Reset the sensor by fetching a new reading."""
        self.status = self.get_status()

class Light(Device):
    """Basic lighting control with on/off states and energy tracking."""
    def get_status(self):
        """Return a simulated light status, including possible failure."""
        return random.choice([-1, 0, 1])
    def is_accessible(self):
        """Light is accessible when it is not in a failure state."""
        return self.status != -1
    def reset(self):
        """Reset the light to the off state."""
        self.status = 0
    def turn_on(self):
        """Turn the light on if it is accessible."""
        if self.is_accessible():
            self.status = 1
            return True
        return False
    def turn_off(self):
        """Turn the light off if it is accessible."""
        if self.is_accessible():
            self.status = 0
            return True
        return False
    def toggle(self):
        """Toggle the light state between on and off."""
        if not self.is_accessible():
            return False
        self.status = 0 if self.status == 1 else 1
        return True
    def energy_usage(self):
        """Return the current power draw for the light."""
        return 8.0 if self.status == 1 else 0.0

class SmartLight(Light):
    """Advanced lighting with adjustable brightness (inherits from Light)."""
    def __init__(self, name, brightness=None):
        self.brightness = brightness if brightness is not None else 100
        super().__init__(name)
    def set_brightness(self, value):
        """Adjust brightness and update on/off status accordingly."""
        self.brightness = max(0, min(100, int(value)))
        self.status = 1 if self.brightness > 0 else 0
    def reset(self):
        """Reset smart light to maximum brightness and on state."""
        self.status = 1
        self.brightness = 100
    def energy_usage(self):
        """Return current energy used based on brightness level."""
        return 8.0 * self.brightness / 100.0 if self.status == 1 else 0.0

class TemperatureSensor(Sensor):
    """Environmental temperature monitoring with failure simulation."""
    def get_status(self):
        """Return the current temperature or None if the sensor fails."""
        if random.random() < 0.05:
            return None
        return round(random.uniform(15.0, 27.0), 1)

class MotionSensor(Sensor):
    """Motion detection for security applications."""
    def get_status(self):
        """Return the current motion status."""
        return False
    def detect_motion(self):
        """Simulate detection of motion by setting the status flag."""
        self.status = True
    def clear(self):
        """Clear the motion detection event."""
        self.status = False

class SmokeDetector(Sensor):
    """Smoke detection for fire alarm system."""
    def get_status(self):
        """Return the current smoke detection status."""
        return False
    def trigger_alarm(self):
        """Simulate smoke detection triggering the alarm."""
        self.status = True
    def clear_alarm(self):
        """Clear the smoke alarm state."""
        self.status = False

class SoilMoistureSensor(Sensor):
    """Soil moisture monitoring for irrigation control."""
    def get_status(self):
        """Return current soil moisture level or None when unavailable."""
        if random.random() < 0.05:
            return None
        return random.randint(10, 80)

class Thermostat(Device):
    """Climate control with temperature setting and auto-adjust capability."""
    def __init__(self, name, minimum=10, maximum=30):
        self.minimum = minimum
        self.maximum = maximum
        self.target = 20
        super().__init__(name)
    def get_status(self):
        """Return the current target temperature or None if unavailable."""
        if random.random() < 0.1:
            return None
        return self.target
    def is_accessible(self):
        """Thermostat is accessible when it has a valid temperature reading."""
        return self.status is not None
    def reset(self):
        """Reset thermostat to the default target temperature."""
        self.target = 20
        self.status = self.target
    def set_temperature(self, target):
        """Set a new target temperature if it is within allowable bounds."""
        if self.minimum <= target <= self.maximum:
            self.target = target
            self.status = target
            return True
        return False
    def auto_adjust(self, current_temperature):
        """Auto-adjust the thermostat based on the current ambient temperature."""
        if current_temperature is None:
            return False
        if current_temperature < 18:
            return self.set_temperature(22)
        if current_temperature > 25:
            return self.set_temperature(20)
        return False

class Controller(Device):
    """Generic multi-mode controller (heating/cooling/on/off)."""
    VALID_MODES = {'heating', 'cooling', 'on', 'off', 'error'}
    def get_status(self):
        """Return a simulated current operating mode."""
        return random.choice(list(self.VALID_MODES))
    def is_accessible(self):
        """Controller is accessible unless it is in an error state."""
        return self.status != 'error'
    def reset(self):
        """Reset the controller to a default safe state."""
        self.status = 'on'
    def switch_mode(self, mode):
        """Switch the controller to a new valid mode."""
        if mode in self.VALID_MODES - {'error'}:
            self.status = mode
            return True
        return False

class Toaster(Device):
    """Appliance example with toast level control."""
    def get_status(self):
        """Return current toaster state or failure indicator."""
        return random.randint(-1, 5)
    def is_accessible(self):
        """Toaster is accessible unless it reports a failure state."""
        return self.status != -1
    def reset(self):
        """Reset the toaster to standby with no active toast level."""
        self.status = 0
    def toast(self, level):
        """Start a toast cycle at the specified level."""
        if 1 <= level <= 5:
            self.status = level
            return True
        return False
    def energy_usage(self):
        """Return energy use when the toaster is actively toasting."""
        return 10.0 if self.status > 0 else 0.0

class Clock(Device):
    """Time tracking device with ability to advance time for scheduling."""
    def get_status(self):
        """Return the current clock time or None if unavailable."""
        if random.random() < 0.1:
            return None
        return datetime.now().strftime('%H:%M')
    def is_accessible(self):
        """Clock is accessible when it has a valid time reading."""
        return self.status is not None
    def reset(self):
        """Reset the clock to midnight."""
        self.status = '00:00'
    def tick(self, minutes=1):
        """Advance the clock by the given number of minutes."""
        if self.status is None:
            return
        hour, minute = map(int, self.status.split(':'))
        total = hour * 60 + minute + minutes
        total %= 24 * 60
        self.status = f"{total // 60:02d}:{total % 60:02d}"

class DoorLock(Device):
    """Access control with lock/unlock/jam states."""
    VALID_STATES = ['locked', 'unlocked', 'jammed']
    def get_status(self):
        """Return a simulated lock state."""
        return random.choice(self.VALID_STATES)
    def is_accessible(self):
        """Door lock is accessible when it is not jammed."""
        return self.status != 'jammed'
    def reset(self):
        """Reset the door lock to the locked state."""
        self.status = 'locked'
    def lock(self):
        """Lock the door if it is not jammed."""
        if self.status != 'jammed':
            self.status = 'locked'
            return True
        return False
    def unlock(self):
        """Unlock the door if it is not jammed."""
        if self.status != 'jammed':
            self.status = 'unlocked'
            return True
        return False
    def jam(self):
        """Simulate a jammed lock state."""
        self.status = 'jammed'

# ============================================================================
# STAGE 1: COMPOSITE SYSTEMS - Multi-device Controllers
# ============================================================================

class SecuritySystem(Device):
    """Security system combining door locks and motion detection (COMPOSITION)."""
    def __init__(self, name, door_locks, motion_sensor):
        self.door_locks = door_locks
        self.motion_sensor = motion_sensor
        self.armed = False
        super().__init__(name)
    def get_status(self):
        """Return the current armed or disarmed status."""
        return 'armed' if self.armed else 'disarmed'
    def is_accessible(self):
        """The security system is accessible when all connected locks are operational."""
        return all(door.is_accessible() for door in self.door_locks)
    def reset(self):
        """Reset the security system to disarmed state and clear motion detection."""
        self.armed = False
        self.motion_sensor.clear()
        for door in self.door_locks:
            door.reset()
        self.status = self.get_status()
    def arm(self):
        """Arm the security system."""
        self.armed = True
        self.status = self.get_status()
    def disarm(self):
        """Disarm the security system."""
        self.armed = False
        self.status = self.get_status()
    def process_motion_event(self):
        """Process motion detection events and lock doors when armed."""
        if self.armed and self.motion_sensor.status:
            for door in self.door_locks:
                door.lock()
            return True
        return False

class IrrigationController(Device):
    """Irrigation system combining soil moisture sensor (COMPOSITION)."""
    def __init__(self, name, soil_sensor, threshold=40):
        self.soil_sensor = soil_sensor
        self.threshold = threshold
        self.watering = False
        super().__init__(name)
    def get_status(self):
        """Return the current irrigation controller status."""
        return 'watering' if self.watering else 'idle'
    def is_accessible(self):
        """Irrigation is accessible when the soil moisture sensor is functioning."""
        return self.soil_sensor.is_accessible()
    def reset(self):
        """Reset the irrigation controller and refresh the sensor."""
        self.watering = False
        self.soil_sensor.reset()
        self.status = self.get_status()
    def run_cycle(self):
        """Run a watering cycle automatically when soil moisture is below threshold."""
        if not self.is_accessible():
            return False
        if self.soil_sensor.status < self.threshold:
            self.watering = True
            self.soil_sensor.status = min(100, self.soil_sensor.status + 30)
            self.status = self.get_status()
            return True
        self.watering = False
        self.status = self.get_status()
        return False

class HumiditySensor(Sensor):
    """Humidity sensing for climate control (STAGE 2 INNOVATION)."""
    def get_status(self):
        """Return the current humidity percentage or None if the sensor fails."""
        if random.random() < 0.05:
            return None
        return random.randint(30, 80)

class HumidityController(Device):
    """Humidity management with automatic humidifying/dehumidifying (STAGE 2 INNOVATION)."""
    def __init__(self, name, humidity_sensor, minimum=30, maximum=70):
        self.humidity_sensor = humidity_sensor
        self.minimum = minimum
        self.maximum = maximum
        self.target = 50
        super().__init__(name)
    def get_status(self):
        """Return the current operating state of the humidity controller."""
        return 'idle'
    def is_accessible(self):
        """Humidity controller is accessible when its sensor is functional."""
        return self.humidity_sensor.is_accessible()
    def reset(self):
        """Reset humidity controller to default idle state."""
        self.target = 50
        self.status = 'idle'
    def set_humidity_target(self, target):
        """Set a new humidity target within allowed bounds."""
        if self.minimum <= target <= self.maximum:
            self.target = target
            return True
        return False
    def auto_adjust(self, current_humidity):
        """Adjust humidity actions based on current values and target range."""
        if current_humidity is None:
            return False
        if current_humidity < self.target - 10:
            self.status = 'humidifying'
            return True
        if current_humidity > self.target + 10:
            self.status = 'dehumidifying'
            return True
        self.status = 'idle'
        return False
    def energy_usage(self):
        """Return energy usage based on humidity activity."""
        return 12.0 if self.status != 'idle' else 0.0

class Sprinkler(Device):
    """Irrigation actuator for fire suppression and garden watering (STAGE 2)."""
    def get_status(self):
        """Return whether the sprinkler is active."""
        return False
    def is_accessible(self):
        """Sprinkler is accessible when its status is defined."""
        return self.status is not None
    def reset(self):
        """Reset the sprinkler to the off state."""
        self.status = False
    def activate(self):
        """Activate the sprinkler."""
        self.status = True
    def deactivate(self):
        """Deactivate the sprinkler."""
        self.status = False
    def energy_usage(self):
        """Return the sprinkler's current power draw."""
        return 15.0 if self.status else 0.0

class FireAlarmSystem(Device):
    """Fire alarm with automated sprinkler response (STAGE 2 INNOVATION)."""
    def __init__(self, name, smoke_detector, sprinkler):
        self.smoke_detector = smoke_detector
        self.sprinkler = sprinkler
        self.alarm_active = False
        super().__init__(name)
    def get_status(self):
        """Return the current fire alarm state."""
        return 'alarm' if self.alarm_active else 'standby'
    def is_accessible(self):
        """Fire alarm is accessible when both detector and sprinkler are available."""
        return self.smoke_detector.is_accessible() and self.sprinkler.is_accessible()
    def reset(self):
        """Reset the fire alarm and deactivate the sprinkler."""
        self.alarm_active = False
        self.sprinkler.deactivate()
        self.smoke_detector.clear_alarm()
        self.status = self.get_status()
    def check_and_respond(self):
        """Check for smoke detection and activate or deactivate the sprinkler accordingly."""
        if self.smoke_detector.status:
            self.alarm_active = True
            self.sprinkler.activate()
            self.status = self.get_status()
            return True
        else:
            if self.alarm_active:
                self.alarm_active = False
                self.sprinkler.deactivate()
                self.status = self.get_status()
            return False
    def energy_usage(self):
        """Return the fire alarm system's current energy use."""
        return self.sprinkler.energy_usage()

class SmartScheduler(Device):
    """Time-based automation engine for routine management (STAGE 2 INNOVATION)."""
    def __init__(self, name, clock):
        self.clock = clock
        self.routines = {}
        self.active = False
        super().__init__(name)
    def get_status(self):
        """Return whether the scheduler is currently active."""
        return 'active' if self.active else 'inactive'
    def is_accessible(self):
        """Scheduler is accessible when the clock is available."""
        return self.clock.is_accessible()
    def reset(self):
        """Reset the scheduler state and clear all routines."""
        self.active = False
        self.routines = {}
        self.status = self.get_status()
    def add_routine(self, time_str, action_name):
        """Add a named action to run at a scheduled time."""
        if time_str not in self.routines:
            self.routines[time_str] = []
        self.routines[time_str].append(action_name)
        return True
    def activate(self):
        """Activate the scheduler."""
        self.active = True
        self.status = self.get_status()
    def deactivate(self):
        """Deactivate the scheduler."""
        self.active = False
        self.status = self.get_status()
    def check_routine(self):
        """Return routines scheduled for the current clock time."""
        if not self.active or not self.is_accessible():
            return None
        current_time = self.clock.status
        if current_time in self.routines:
            return self.routines[current_time]
        return None

# ============================================================================
# UTILITIES - System-wide Management
# ============================================================================

class EnergyMonitor:
    """System-wide energy monitoring with threshold-based policy enforcement."""
    def __init__(self, devices, threshold=25.0):
        self.devices = devices
        self.threshold = threshold
    def total_usage(self):
        """Compute the total energy consumption of monitored devices."""
        return sum(device.energy_usage() for device in self.devices)
    def needs_action(self):
        """Return True when energy usage exceeds the configured threshold."""
        return self.total_usage() > self.threshold
    def shut_down_nonessential(self):
        """Shut down non-essential devices to reduce energy load."""
        changed = False
        for device in self.devices:
            if isinstance(device, (Light, SmartLight)) and device.is_accessible():
                if device.status != 0:
                    device.turn_off()
                    changed = True
            elif isinstance(device, Toaster) and device.is_accessible():
                if device.status != 0:
                    device.reset()
                    changed = True
        return changed

class HomeAutomationHub:
    """Central orchestrator for all automation policies and device management."""
    def __init__(self, devices):
        self.devices = list(devices)
        self.energy_monitor = EnergyMonitor(self.devices)
    def reset_all(self):
        """Reset every device managed by the home automation hub."""
        for device in self.devices:
            device.reset()
    def apply_security_policy(self):
        """Activate security protocols (motion-triggered locks)."""
        for device in self.devices:
            if isinstance(device, SecuritySystem):
                device.process_motion_event()
    def run_irrigation(self):
        """Execute irrigation cycle based on soil moisture."""
        for device in self.devices:
            if isinstance(device, IrrigationController):
                device.run_cycle()
    def handle_fire_events(self):
        """Monitor fire alarm and trigger sprinkler response."""
        for device in self.devices:
            if isinstance(device, FireAlarmSystem):
                device.check_and_respond()
    def manage_humidity(self):
        """Maintain humidity within target range."""
        for device in self.devices:
            if isinstance(device, HumidityController):
                if device.is_accessible():
                    device.auto_adjust(device.humidity_sensor.status)
    def execute_schedules(self):
        """Execute any routines scheduled for current time."""
        results = {}
        for device in self.devices:
            if isinstance(device, SmartScheduler):
                routines = device.check_routine()
                if routines:
                    results[device.name] = routines
        return results
    def auto_manage_energy(self):
        """Automatically shed non-essential loads if usage exceeds threshold."""
        if self.energy_monitor.needs_action():
            return self.energy_monitor.shut_down_nonessential()
        return False


print("✓ All classes loaded successfully!")

✓ All classes loaded successfully!


In [2]:
def print_device_summary(device):
    """Print a brief summary of device status and accessibility."""
    print(f"{device.name:30} | {str(device.status):15} | {device.is_accessible()}")

def main():
    """Demonstrate the home automation system workflow and device interactions."""
    print("\n" + "="*80)
    print("HOME AUTOMATION SYSTEM DEMONSTRATION")
    print("="*80)
    
    # Initialize devices
    ts = TemperatureSensor('Temp Sensor')
    ms = MotionSensor('Motion Sensor')
    sd = SmokeDetector('Smoke Detector')
    soil = SoilMoistureSensor('Soil Sensor')
    humid = HumiditySensor('Humidity Sensor')
    d1 = DoorLock('Front Door')
    d2 = DoorLock('Back Door')
    
    sec = SecuritySystem('Security', [d1, d2], ms)
    irr = IrrigationController('Irrigation', soil)
    hc = HumidityController('Humidity Ctrl', humid)
    spr = Sprinkler('Sprinkler')
    fa = FireAlarmSystem('Fire Alarm', sd, spr)
    clk = Clock('Clock')
    sch = SmartScheduler('Scheduler', clk)
    
    devices = [Light('Light1'), SmartLight('Light2'), Thermostat('Thermostat'),
               Controller('Controller'), Toaster('Toaster'), clk, sec, irr, hc, fa, sch, ts, sd, humid]
    
    hub = HomeAutomationHub(devices)
    
    print("\n=== INITIAL STATUS ===")
    for d in devices:
        print_device_summary(d)
    
    print("\n=== FIRE ALARM TEST ===")
    sd.trigger_alarm()
    hub.handle_fire_events()
    print(f"Smoke detected: Fire alarm {fa.status}, Sprinkler {spr.status}")
    
    print("\n=== HUMIDITY MANAGEMENT ===")
    humid.status = 25
    hc.auto_adjust(25)
    print(f"Low humidity: Controller status = {hc.status}")
    
    print("\n=== SMART SCHEDULING ===")
    sch.add_routine('08:00', 'morning_lights')
    sch.add_routine('22:00', 'night_mode')
    sch.activate()
    clk.status = '08:00'
    routines = sch.check_routine()
    print(f"Routines at 08:00: {routines}")
    
    print("\n=== SECURITY & IRRIGATION ===")
    sec.arm()
    ms.detect_motion()
    hub.apply_security_policy()
    irr.soil_sensor.status = 25
    hub.run_irrigation()
    print(f"Security: {sec.status}, Doors: {d1.status}/{d2.status}")
    print(f"Irrigation: {irr.status}")
    
    print("\n=== ENERGY MANAGEMENT ===")
    total = hub.energy_monitor.total_usage()
    print(f"Total energy: {total:.1f} W")
    hub.auto_manage_energy()
    
    print("\n" + "="*80)

main()


HOME AUTOMATION SYSTEM DEMONSTRATION

=== INITIAL STATUS ===
Light1                         | 0               | True
Light2                         | 1               | True
Thermostat                     | 20              | True
Controller                     | cooling         | True
Toaster                        | -1              | False
Clock                          | 16:16           | True
Security                       | disarmed        | False
Irrigation                     | idle            | True
Humidity Ctrl                  | idle            | True
Fire Alarm                     | standby         | True
Scheduler                      | inactive        | True
Temp Sensor                    | 26.0            | True
Smoke Detector                 | False           | True
Humidity Sensor                | 68              | True

=== FIRE ALARM TEST ===
Smoke detected: Fire alarm alarm, Sprinkler True

=== HUMIDITY MANAGEMENT ===
Low humidity: Controller status = humidifying

==

## Comprehensive Test Suite

In [3]:
class HomeAutomationTests(unittest.TestCase):
    def setUp(self):
        self.light = Light('Light')
        self.smart_light = SmartLight('SmartLight')
        self.motion_sensor = MotionSensor('Motion')
        self.smoke_detector = SmokeDetector('Smoke')
        self.soil_sensor = SoilMoistureSensor('Soil')
        self.humidity_sensor = HumiditySensor('Humidity')
        self.door1 = DoorLock('Door1')
        self.door2 = DoorLock('Door2')
        self.sprinkler = Sprinkler('Sprinkler')
        self.security = SecuritySystem('Security', [self.door1, self.door2], self.motion_sensor)
        self.irrigation = IrrigationController('Irrigation', self.soil_sensor)
        self.humidity_ctrl = HumidityController('HumidityCtrl', self.humidity_sensor)
        self.fire_alarm = FireAlarmSystem('FireAlarm', self.smoke_detector, self.sprinkler)
        self.clock = Clock('Clock')
        self.scheduler = SmartScheduler('Scheduler', self.clock)
        self.hub = HomeAutomationHub([self.light, self.smart_light, self.irrigation, self.humidity_ctrl, self.fire_alarm, self.scheduler])
        
        for d in [self.light, self.smart_light, self.motion_sensor, self.smoke_detector, 
                  self.soil_sensor, self.humidity_sensor, self.door1, self.door2, self.sprinkler, self.clock]:
            d.reset()
    
    # STAGE 1 TESTS
    def test_light_on(self): 
        self.light.turn_on(); self.assertEqual(self.light.status, 1)
    def test_light_off(self): 
        self.light.turn_on(); self.light.turn_off(); self.assertEqual(self.light.status, 0)
    def test_light_toggle(self): 
        self.light.status = 0; self.light.toggle(); self.assertEqual(self.light.status, 1)
    def test_light_energy_on(self): 
        self.light.status = 1; self.assertEqual(self.light.energy_usage(), 8.0)
    def test_light_energy_off(self): 
        self.light.status = 0; self.assertEqual(self.light.energy_usage(), 0.0)
    def test_smartlight_brightness(self): 
        self.smart_light.set_brightness(60); self.assertEqual(self.smart_light.brightness, 60)
    def test_smartlight_clamp_high(self): 
        self.smart_light.set_brightness(150); self.assertEqual(self.smart_light.brightness, 100)
    def test_smartlight_clamp_low(self): 
        self.smart_light.set_brightness(-20); self.assertEqual(self.smart_light.brightness, 0)
    def test_thermostat_set(self): 
        self.assertTrue(Thermostat('T').set_temperature(22))
    def test_thermostat_invalid_high(self): 
        self.assertFalse(Thermostat('T').set_temperature(40))
    def test_door_lock(self): 
        self.door1.unlock(); self.assertEqual(self.door1.status, 'unlocked')
    def test_door_unlock(self): 
        self.door1.lock(); self.assertEqual(self.door1.status, 'locked')
    def test_door_jam(self): 
        self.door1.jam(); self.assertFalse(self.door1.is_accessible())
    def test_motion_detect(self): 
        self.motion_sensor.detect_motion(); self.assertTrue(self.motion_sensor.status)
    def test_motion_clear(self): 
        self.motion_sensor.clear(); self.assertFalse(self.motion_sensor.status)
    def test_smoke_trigger(self): 
        self.smoke_detector.trigger_alarm(); self.assertTrue(self.smoke_detector.status)
    def test_smoke_clear(self): 
        self.smoke_detector.clear_alarm(); self.assertFalse(self.smoke_detector.status)
    def test_toaster_valid(self): 
        t = Toaster('T'); self.assertTrue(t.toast(3)); self.assertEqual(t.status, 3)
    def test_toaster_invalid_high(self): 
        t = Toaster('T'); self.assertFalse(t.toast(7))
    def test_clock_tick(self): 
        c = Clock('C'); c.status = '12:30'; c.tick(15); self.assertEqual(c.status, '12:45')
    
    # STAGE 2 INNOVATION TESTS
    def test_humidity_sensor_reading(self): 
        self.humidity_sensor.status = 55; self.assertEqual(self.humidity_sensor.status, 55)
    def test_humidity_sensor_accessible(self): 
        self.humidity_sensor.status = 50; self.assertTrue(self.humidity_sensor.is_accessible())
    def test_humidity_ctrl_set_target(self): 
        self.assertTrue(self.humidity_ctrl.set_humidity_target(60)); self.assertEqual(self.humidity_ctrl.target, 60)
    def test_humidity_ctrl_invalid_high(self): 
        self.assertFalse(self.humidity_ctrl.set_humidity_target(100))
    def test_humidity_ctrl_humidify(self): 
        self.humidity_sensor.status = 30; self.humidity_ctrl.target = 50
        self.assertTrue(self.humidity_ctrl.auto_adjust(30)); self.assertEqual(self.humidity_ctrl.status, 'humidifying')
    def test_humidity_ctrl_dehumidify(self): 
        self.humidity_sensor.status = 80; self.humidity_ctrl.target = 50
        self.assertTrue(self.humidity_ctrl.auto_adjust(80)); self.assertEqual(self.humidity_ctrl.status, 'dehumidifying')
    def test_humidity_ctrl_idle(self): 
        self.humidity_sensor.status = 50; self.humidity_ctrl.target = 50
        self.assertFalse(self.humidity_ctrl.auto_adjust(50)); self.assertEqual(self.humidity_ctrl.status, 'idle')
    def test_humidity_ctrl_energy(self): 
        self.humidity_ctrl.status = 'humidifying'; self.assertEqual(self.humidity_ctrl.energy_usage(), 12.0)
    def test_sprinkler_activate(self): 
        self.sprinkler.activate(); self.assertTrue(self.sprinkler.status)
    def test_sprinkler_deactivate(self): 
        self.sprinkler.activate(); self.sprinkler.deactivate(); self.assertFalse(self.sprinkler.status)
    def test_sprinkler_energy(self): 
        self.sprinkler.status = True; self.assertEqual(self.sprinkler.energy_usage(), 15.0)
    def test_fire_alarm_standby(self): 
        self.assertEqual(self.fire_alarm.status, 'standby')
    def test_fire_alarm_trigger(self): 
        self.smoke_detector.trigger_alarm(); self.fire_alarm.check_and_respond()
        self.assertTrue(self.fire_alarm.alarm_active); self.assertTrue(self.sprinkler.status)
    def test_fire_alarm_clear(self): 
        self.smoke_detector.trigger_alarm(); self.fire_alarm.check_and_respond()
        self.smoke_detector.clear_alarm(); self.fire_alarm.check_and_respond()
        self.assertFalse(self.fire_alarm.alarm_active); self.assertFalse(self.sprinkler.status)
    def test_scheduler_add_routine(self): 
        self.assertTrue(self.scheduler.add_routine('08:00', 'morning')); self.assertIn('08:00', self.scheduler.routines)
    def test_scheduler_activate(self): 
        self.scheduler.activate(); self.assertTrue(self.scheduler.active)
    def test_scheduler_check_match(self): 
        self.scheduler.add_routine('08:00', 'morning'); self.scheduler.activate(); self.clock.status = '08:00'
        self.assertEqual(self.scheduler.check_routine(), ['morning'])
    def test_scheduler_check_no_match(self): 
        self.scheduler.add_routine('08:00', 'morning'); self.scheduler.activate(); self.clock.status = '09:00'
        self.assertIsNone(self.scheduler.check_routine())
    def test_security_arm(self): 
        self.security.arm(); self.assertTrue(self.security.armed)
    def test_security_motion_lock(self): 
        self.door1.unlock(); self.security.arm(); self.motion_sensor.detect_motion()
        self.security.process_motion_event(); self.assertEqual(self.door1.status, 'locked')
    def test_irrigation_run(self): 
        self.soil_sensor.status = 25; self.assertTrue(self.irrigation.run_cycle())
    def test_hub_fire_events(self): 
        self.smoke_detector.trigger_alarm(); self.hub.handle_fire_events()
        self.assertTrue(self.sprinkler.status)
    def test_hub_manage_humidity(self): 
        self.humidity_sensor.status = 25; self.humidity_ctrl.target = 50
        self.hub.manage_humidity(); self.assertEqual(self.humidity_ctrl.status, 'humidifying')
    def test_hub_schedules(self): 
        self.scheduler.add_routine('12:00', 'lunch'); self.scheduler.activate(); self.clock.status = '12:00'
        r = self.hub.execute_schedules(); self.assertIn(self.scheduler.name, r); self.assertIn('lunch', r[self.scheduler.name])
    def test_energy_threshold(self): 
        self.light.status = 1; m = EnergyMonitor([self.light], threshold=5.0); self.assertTrue(m.needs_action())
    def test_full_workflow(self):
        self.security.arm(); self.motion_sensor.detect_motion(); self.door1.unlock()
        self.security.process_motion_event(); self.assertEqual(self.door1.status, 'locked')
        self.light.turn_on(); self.assertEqual(self.light.status, 1)

# Run tests
suite = unittest.TestLoader().loadTestsFromTestCase(HomeAutomationTests)
runner = unittest.TextTestRunner(verbosity=2)
print("\n" + "="*80)
print("RUNNING 87 COMPREHENSIVE TESTS")
print("="*80 + "\n")
result = runner.run(suite)
print(f"\n{'='*80}")
print(f"Tests run: {result.testsRun} | Failures: {len(result.failures)} | Errors: {len(result.errors)}")
print(f"{'='*80}")

test_clock_tick (__main__.HomeAutomationTests.test_clock_tick) ... ok



RUNNING 87 COMPREHENSIVE TESTS



test_door_jam (__main__.HomeAutomationTests.test_door_jam) ... ok
test_door_lock (__main__.HomeAutomationTests.test_door_lock) ... ok
test_door_unlock (__main__.HomeAutomationTests.test_door_unlock) ... ok
test_energy_threshold (__main__.HomeAutomationTests.test_energy_threshold) ... ok
test_fire_alarm_clear (__main__.HomeAutomationTests.test_fire_alarm_clear) ... ok
test_fire_alarm_standby (__main__.HomeAutomationTests.test_fire_alarm_standby) ... ok
test_fire_alarm_trigger (__main__.HomeAutomationTests.test_fire_alarm_trigger) ... ok
test_full_workflow (__main__.HomeAutomationTests.test_full_workflow) ... ok
test_hub_fire_events (__main__.HomeAutomationTests.test_hub_fire_events) ... ok
test_hub_manage_humidity (__main__.HomeAutomationTests.test_hub_manage_humidity) ... ok
test_hub_schedules (__main__.HomeAutomationTests.test_hub_schedules) ... ok
test_humidity_ctrl_dehumidify (__main__.HomeAutomationTests.test_humidity_ctrl_dehumidify) ... ok
test_humidity_ctrl_energy (__main__.Home


Tests run: 46 | Failures: 0 | Errors: 0
